# 03 — Análise exploratória independente: Occupancy Detection

Estatísticas descritivas, frequência de ocupação, gráficos por hora/dia e correlações entre variáveis. Este notebook **não depende dos resultados de clusterização**.

**Arquivos necessários:** `datatest.txt`, `datatraining.txt` e `datatest2.txt` na pasta de execução (ou edite `ARQ_X0`, `ARQ_CAL` e `ARQ_TESTE`). Execute as células na ordem apresentada.

**Observação:** há dois gráficos de correlação herdados do notebook original. O último utiliza apenas a amostra inicial (`base`) e deve ser preferido para decisões de pré-processamento sem informações futuras.

In [ ]:
# %% 0. Imports e configuração
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# True = grades pequenas, só para testar se o código roda de ponta a ponta
RAPIDO = False

# Caminhos dos arquivos (ajuste se necessário)
ARQ_X0 = "datatest.txt"
ARQ_CAL = "datatraining.txt"
ARQ_TESTE = "datatest2.txt"

# Variáveis usadas na clusterização
COLS = [
    "Temperature",
    "Light",
    "CO2",
    "Humidity"
]

TAM_LOTE = 60          # 1 ponto = 1 minuto, então 60 = 1 hora
JANELA_ARI = 4320      # 3 dias (janela que sempre contém as duas classes)


In [ ]:
# %% 1. Leitura e junção dos arquivos
partes = []
for arq in [ARQ_X0, ARQ_CAL, ARQ_TESTE]:
    d = pd.read_csv(arq, index_col=0)   # 1ª coluna é um índice sem nome no cabeçalho
    d["origem"] = arq
    partes.append(d)

df = pd.concat(partes)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Formato:", df.shape, "(esperado: 20560 linhas)")
print("Ordenado no tempo:", df["date"].is_monotonic_increasing)
print("Nulos:", int(df.isna().sum().sum()))
print(df.groupby("origem")["date"].agg(["min", "max", "count"]))


In [ ]:
# Preparação exclusiva da análise descritiva — independente da calibração
base = df[df["origem"] == ARQ_X0].copy()
cal = df[df["origem"] == ARQ_CAL].copy()
teste = df[df["origem"] == ARQ_TESTE].copy()

dados_eda = df.sort_values("date").reset_index(drop=True).copy()
dados_eda["hora"] = dados_eda["date"].dt.hour
dados_eda["dia_semana"] = dados_eda["date"].dt.dayofweek
nomes_dias = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
dados_eda["dia_nome"] = dados_eda["dia_semana"].map(dict(enumerate(nomes_dias)))
COLS_EDA = ["Temperature", "Light", "CO2", "Humidity", "HumidityRatio"]
print("Registros:", len(dados_eda), "| Valores ausentes:", int(dados_eda.isna().sum().sum()))

In [ ]:
# ============================================================
# OCCUPANCY DETECTION
# ANÁLISE EXPLORATÓRIA + DIAGNÓSTICO FINAL
#
# Executar após o experimento anterior.
# Não modifica os algoritmos nem suas predições.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

COLS_EDA = [
    "Temperature", "Light", "CO2",
    "Humidity", "HumidityRatio"
]


In [ ]:
# ============================================================
# 2. DESCRIÇÃO GERAL
# ============================================================

print("\n=== Estatísticas descritivas ===")

descricao = dados_eda[COLS_EDA].describe().T

print(
    descricao.round(3).to_string()
)

print("\n=== Ocupação por conjunto ===")

tabela_ocupacao = []

for nome, dados in [
    ("Inicialização", base),
    ("Calibração", cal),
    ("Teste", teste)
]:

    tabela_ocupacao.append({
        "Conjunto": nome,
        "N": len(dados),
        "Desocupada": int(
            (dados["Occupancy"] == 0).sum()
        ),
        "Ocupada": int(
            (dados["Occupancy"] == 1).sum()
        ),
        "Proporção ocupada": (
            dados["Occupancy"].mean()
        )
    })

tabela_ocupacao = pd.DataFrame(
    tabela_ocupacao
)

print(
    tabela_ocupacao.round(4).to_string(
        index=False
    )
)


In [ ]:
# ============================================================
# 3. OCUPAÇÃO POR HORA
# ============================================================

ocupacao_hora = (
    dados_eda.groupby("hora")["Occupancy"]
    .agg(["mean", "size"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    ocupacao_hora["hora"],
    ocupacao_hora["mean"]
)

ax.set_xticks(range(24))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Proporção de ocupação")
ax.set_title("Ocupação da sala por hora")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 4. OCUPAÇÃO POR DIA DA SEMANA
# ============================================================

ocupacao_dia = (
    dados_eda.groupby("dia_semana")["Occupancy"]
    .agg(["mean", "size"])
    .reindex(range(7))
)

fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(
    nomes_dias,
    ocupacao_dia["mean"]
)

ax.set_xlabel("Dia da semana")
ax.set_ylabel("Proporção de ocupação")
ax.set_title("Ocupação por dia da semana")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 5. MAPA DE CALOR — DIA X HORA
# ============================================================

matriz_ocupacao = dados_eda.pivot_table(
    index="dia_semana",
    columns="hora",
    values="Occupancy",
    aggfunc="mean"
).reindex(
    index=range(7),
    columns=range(24)
)

fig, ax = plt.subplots(figsize=(12, 4.5))

im = ax.imshow(
    matriz_ocupacao.to_numpy(),
    aspect="auto",
    vmin=0,
    vmax=1,
    cmap="YlOrRd"
)

ax.set_yticks(range(7))
ax.set_yticklabels(nomes_dias)
ax.set_xticks(range(24))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Dia da semana")
ax.set_title("Proporção de ocupação por dia e hora")

plt.colorbar(
    im,
    ax=ax,
    label="Proporção de ocupação"
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 6. DISTRIBUIÇÃO DAS VARIÁVEIS POR OCUPAÇÃO
# ============================================================

fig, axes = plt.subplots(
    1, len(COLS_EDA),
    figsize=(16, 4)
)

for ax, coluna in zip(axes, COLS_EDA):

    valores = [
        dados_eda.loc[
            dados_eda["Occupancy"] == classe,
            coluna
        ].to_numpy()
        for classe in [0, 1]
    ]

    ax.boxplot(
        valores,
        tick_labels=["Não", "Sim"],
        showfliers=False
    )

    ax.set_title(coluna)
    ax.set_xlabel("Ocupada")
    ax.grid(axis="y", alpha=0.2)

plt.suptitle(
    "Distribuição dos atributos por estado de ocupação"
)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7. CORRELAÇÃO ENTRE OS ATRIBUTOS
# ============================================================

corr = dados_eda[COLS_EDA].corr()

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    corr,
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(range(len(COLS_EDA)))
ax.set_yticks(range(len(COLS_EDA)))

ax.set_xticklabels(
    COLS_EDA,
    rotation=45,
    ha="right"
)
ax.set_yticklabels(COLS_EDA)

for i in range(len(COLS_EDA)):
    for j in range(len(COLS_EDA)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

plt.colorbar(im, ax=ax)
ax.set_title("Correlação entre os atributos")

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 7. CORRELAÇÃO ENTRE OS ATRIBUTOS — APENAS INICIALIZAÇÃO
# ============================================================

COLS_EDA = [
    "Temperature", "Light", "CO2",
    "Humidity", "HumidityRatio"
]

# Utilizar somente as observações iniciais (X0)
dados_corr = base[COLS_EDA].copy()

# Matriz de correlação de Pearson
corr = dados_corr.corr(method="pearson")

print(f"Observações utilizadas: {len(dados_corr)}")
print("\nMatriz de correlação:")
print(corr.round(4))

print(
    "\nCorrelação entre Humidity e HumidityRatio:",
    round(corr.loc["Humidity", "HumidityRatio"], 4)
)

# Gráfico
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    corr,
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(range(len(COLS_EDA)))
ax.set_yticks(range(len(COLS_EDA)))

ax.set_xticklabels(
    COLS_EDA,
    rotation=45,
    ha="right"
)
ax.set_yticklabels(COLS_EDA)

for i in range(len(COLS_EDA)):
    for j in range(len(COLS_EDA)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=10
        )

plt.colorbar(im, ax=ax, label="Correlação de Pearson")

ax.set_title(
    "Correlação entre os atributos — Inicialização"
)

plt.tight_layout()

plt.savefig(
    "correlacao_inicializacao.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
